# Primer entrenamiento baseline

Este notebook hace:
1. Escribe `src/training/train.py` (loop de entrenamiento).
2. Escribe `scripts/run_train.py` (entrypoint).
3. Ejecuta el entrenamiento.
4. Plotea las curvas de loss y AUC.

**Runtime:** T4 GPU obligatorio.

**Tiempo estimado:** 3-8 minutos para 10 epochs con 199 imgs de train (batch 32).

**Resultado esperado:** val AUC > 0.70 despues de entrenar (partimos de ~0.34
del modelo con cabeza random). Si supera 0.80 mejor aun.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/Tesis_CXR'
CODE_DIR = f'{PROJECT_ROOT}/code'

import sys
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠ No hay GPU: el entrenamiento sera lento. Cambia a T4.")


In [ ]:
!pip install -q torchxrayvision scipy


## 2. Escribir `src/training/train.py`

In [ ]:
from pathlib import Path

content = r'''"""
src/training/train.py

Loop de entrenamiento reutilizable para el baseline CXR binario.

Caracteristicas:
  - CrossEntropyLoss sobre logits (compatible con num_classes=2 y extensible).
  - Validacion al final de cada epoch con las 5 metricas del charter.
  - Early stopping basado en val AUC (metrica mas robusta al desbalance).
  - Checkpointing: guarda el MEJOR modelo segun val AUC, no el ultimo.
  - Mixed precision (AMP) via torch.amp.autocast para velocidad en T4.
  - History dict con metricas por epoch para plottear curvas.

Uso:
    from src.training.train import train_model
    history = train_model(model, train_loader, val_loader, cfg, device, logger)
"""
from __future__ import annotations

import copy
from pathlib import Path
from typing import Any

import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import DataLoader

from src.evaluation.metrics import compute_metrics, logits_to_probs
from src.training.utils import save_checkpoint


# ---------------------------------------------------------------------------
# Una epoch de entrenamiento
# ---------------------------------------------------------------------------
def train_one_epoch(model: nn.Module, loader: DataLoader,
                    optimizer: torch.optim.Optimizer, criterion: nn.Module,
                    device: torch.device, use_amp: bool,
                    scaler: GradScaler | None) -> dict:
    """Corre una epoch de entrenamiento, devuelve loss y metricas."""
    model.train()
    losses = []
    all_probs = []
    all_labels = []

    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if use_amp and scaler is not None:
            with autocast(device_type='cuda', dtype=torch.float16):
                logits = model(images)
                loss = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

        losses.append(loss.item())
        with torch.no_grad():
            probs = logits_to_probs(logits.detach().float(), positive_class=1)
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    metrics = compute_metrics(all_labels, all_probs, threshold=0.5)
    mean_loss = float(np.mean(losses))

    return {
        'loss': mean_loss,
        'metrics': metrics,
    }


# ---------------------------------------------------------------------------
# Una epoch de validacion
# ---------------------------------------------------------------------------
@torch.no_grad()
def validate_one_epoch(model: nn.Module, loader: DataLoader,
                       criterion: nn.Module, device: torch.device) -> dict:
    """Evalua el modelo en el set de validacion."""
    model.eval()
    losses = []
    all_probs = []
    all_labels = []

    for batch in loader:
        images = batch['image'].to(device, non_blocking=True)
        labels = batch['label'].to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        losses.append(loss.item())
        probs = logits_to_probs(logits.float(), positive_class=1)
        all_probs.append(probs.cpu())
        all_labels.append(labels.cpu())

    all_probs = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()

    metrics = compute_metrics(all_labels, all_probs, threshold=0.5)
    mean_loss = float(np.mean(losses))

    return {
        'loss': mean_loss,
        'metrics': metrics,
    }


# ---------------------------------------------------------------------------
# Loop principal
# ---------------------------------------------------------------------------
def train_model(model: nn.Module, train_loader: DataLoader,
                val_loader: DataLoader, cfg: Any,
                device: torch.device, logger) -> dict:
    """
    Entrena el modelo con early stopping y guarda el mejor checkpoint.

    Args:
        model: nn.Module a entrenar (ya en device).
        train_loader / val_loader: DataLoaders de train y val.
        cfg: objeto Config cargado con load_config.
        device: torch.device.
        logger: logger para escribir traza.

    Returns:
        dict con:
          'history': lista de dicts, uno por epoch.
          'best_epoch': epoch del mejor modelo.
          'best_val_auc': AUC de val del mejor modelo.
          'checkpoint_path': ruta donde se guardo el mejor modelo.
    """
    # Optimizer solo sobre parametros entrenables
    trainable = [p for p in model.parameters() if p.requires_grad]
    logger.info("Parametros entrenables: %d tensors, %d scalars",
                len(trainable), sum(p.numel() for p in trainable))

    optimizer_name = cfg.training.get('optimizer', 'adam').lower()
    if optimizer_name == 'adam':
        optimizer = torch.optim.Adam(
            trainable,
            lr=cfg.training['learning_rate'],
            weight_decay=cfg.training.get('weight_decay', 0.0),
        )
    elif optimizer_name == 'sgd':
        optimizer = torch.optim.SGD(
            trainable,
            lr=cfg.training['learning_rate'],
            momentum=0.9,
            weight_decay=cfg.training.get('weight_decay', 0.0),
        )
    else:
        raise ValueError(f"Optimizer '{optimizer_name}' no soportado")

    criterion = nn.CrossEntropyLoss()

    use_amp = cfg.training.get('use_amp', False) and torch.cuda.is_available()
    scaler = GradScaler() if use_amp else None
    logger.info("Mixed precision (AMP): %s", use_amp)

    epochs = cfg.training['epochs']
    patience = cfg.training.get('patience', 3)

    # Preparar paths de checkpoint
    ckpt_dir = Path(cfg.paths['checkpoints'])
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_ckpt_path = ckpt_dir / 'baseline_best.pt'

    # Tracking de best model
    best_val_auc = -float('inf')
    best_epoch = -1
    best_state_dict = None
    epochs_without_improvement = 0

    history = []

    logger.info("=" * 70)
    logger.info("Iniciando entrenamiento: %d epochs, patience=%d", epochs, patience)
    logger.info("=" * 70)

    for epoch in range(1, epochs + 1):
        # Train
        train_res = train_one_epoch(model, train_loader, optimizer, criterion,
                                    device, use_amp, scaler)
        # Validate
        val_res = validate_one_epoch(model, val_loader, criterion, device)

        tr_m = train_res['metrics']
        va_m = val_res['metrics']

        logger.info(
            "Epoch %2d/%d | train loss=%.4f AUC=%.4f acc=%.4f | "
            "val loss=%.4f AUC=%.4f acc=%.4f sens=%.4f spec=%.4f",
            epoch, epochs,
            train_res['loss'], tr_m.auc, tr_m.accuracy,
            val_res['loss'], va_m.auc, va_m.accuracy, va_m.sensitivity, va_m.specificity,
        )

        history.append({
            'epoch': epoch,
            'train_loss': train_res['loss'],
            'train_metrics': tr_m.as_dict(),
            'val_loss': val_res['loss'],
            'val_metrics': va_m.as_dict(),
        })

        # Check best model (por val AUC)
        # Si AUC es NaN (batch con una sola clase), usamos accuracy como fallback
        current_score = va_m.auc if not np.isnan(va_m.auc) else va_m.accuracy

        if current_score > best_val_auc:
            best_val_auc = current_score
            best_epoch = epoch
            best_state_dict = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
            logger.info("  → Nuevo mejor modelo (val AUC=%.4f)", best_val_auc)
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                logger.info("Early stopping: %d epochs sin mejora en val AUC.",
                            patience)
                break

    # Guardar mejor checkpoint
    if best_state_dict is not None:
        save_checkpoint({
            'epoch': best_epoch,
            'model_state_dict': best_state_dict,
            'best_val_auc': best_val_auc,
            'cfg_snapshot': {
                'backbone': cfg.model['backbone'],
                'num_classes': cfg.model['num_classes'],
                'finetune_mode': cfg.model['finetune_mode'],
            },
        }, best_ckpt_path)
        logger.info("Mejor modelo guardado en: %s (epoch %d, val AUC=%.4f)",
                    best_ckpt_path, best_epoch, best_val_auc)

    logger.info("=" * 70)
    logger.info("Entrenamiento terminado.")

    return {
        'history': history,
        'best_epoch': best_epoch,
        'best_val_auc': best_val_auc,
        'checkpoint_path': str(best_ckpt_path),
    }
'''
path = f'{CODE_DIR}/src/training/train.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 3. Escribir `scripts/run_train.py`

In [ ]:
content = r'''"""
scripts/run_train.py

Entrypoint del entrenamiento baseline.

Hace:
  1. Carga config.
  2. Fija seed.
  3. Construye datasets, loaders, modelo.
  4. Llama a train_model() del src/training/train.py.
  5. Guarda history.json para poder plottear curvas despues.

Uso (desde Colab):
    !python /content/drive/MyDrive/Tesis_CXR/code/scripts/run_train.py
"""
from __future__ import annotations

import json
import sys
from pathlib import Path

import torch
from torch.utils.data import DataLoader

CODE_DIR = Path(__file__).resolve().parent.parent
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from src.datasets.nih import build_nih_datasets
from src.datasets.transforms import build_train_transform, build_eval_transform
from src.models.baseline import build_baseline_model
from src.training.train import train_model
from src.training.utils import (
    get_device,
    get_logger,
    load_config,
    set_seed,
)


def main(config_path: str | None = None) -> None:
    if config_path is None:
        config_path = CODE_DIR / "configs" / "baseline.yaml"
    cfg = load_config(config_path)

    set_seed(cfg.project['seed'])

    log_file = Path(cfg.paths['logs']) / "train.log"
    logger = get_logger("train", log_file=log_file)

    logger.info("=" * 70)
    logger.info("RUN TRAIN - Sprint 1 baseline")
    logger.info("Config: %s", config_path)
    logger.info("=" * 70)

    device = get_device()
    logger.info("Device: %s", device)
    if device.type == 'cuda':
        logger.info("GPU: %s", torch.cuda.get_device_name(0))

    # 1. Datasets y loaders
    train_tfm = build_train_transform(image_size=cfg.preprocessing['image_size'])
    eval_tfm = build_eval_transform(image_size=cfg.preprocessing['image_size'])

    train_ds, val_ds, test_ds = build_nih_datasets(cfg, train_tfm, eval_tfm)
    logger.info("Train: %d | Val: %d | Test: %d",
                len(train_ds), len(val_ds), len(test_ds))

    # Para train queremos shuffle; para val/test no (reproducibilidad).
    train_loader = DataLoader(
        train_ds, batch_size=cfg.training['batch_size'], shuffle=True,
        num_workers=cfg.training.get('num_workers', 0),
        pin_memory=(device.type == 'cuda'),
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.training['batch_size'], shuffle=False,
        num_workers=cfg.training.get('num_workers', 0),
        pin_memory=(device.type == 'cuda'),
    )

    # 2. Modelo
    model = build_baseline_model(cfg).to(device)
    logger.info("Modelo: %s (backbone=%s, mode=%s)",
                model.__class__.__name__,
                cfg.model['backbone'], cfg.model['finetune_mode'])
    logger.info("  total params: %d | trainable: %d",
                model.total_params(), model.trainable_params())

    # 3. Entrenamiento
    result = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        cfg=cfg,
        device=device,
        logger=logger,
    )

    # 4. Guardar history.json
    history_path = Path(cfg.paths['experiments']) / "history.json"
    history_path.parent.mkdir(parents=True, exist_ok=True)
    with history_path.open('w', encoding='utf-8') as f:
        json.dump({
            'history': result['history'],
            'best_epoch': result['best_epoch'],
            'best_val_auc': result['best_val_auc'],
            'checkpoint_path': result['checkpoint_path'],
        }, f, indent=2, ensure_ascii=False)

    logger.info("History guardado en: %s", history_path)
    logger.info("RESUMEN: best epoch=%d | best val AUC=%.4f",
                result['best_epoch'], result['best_val_auc'])


if __name__ == "__main__":
    main()
'''
path = f'{CODE_DIR}/scripts/run_train.py'
Path(path).write_text(content, encoding='utf-8')
print(f'OK -> {path} ({len(content):,} chars)')


## 4. Ejecutar el entrenamiento

Esta celda corre todas las epochs. Veras un log por epoch con loss y metricas
de train y val. Al final, el mejor modelo queda guardado en
`experiments/checkpoints/baseline_best.pt`.

In [ ]:
import subprocess
result = subprocess.run(
    ['python', f'{CODE_DIR}/scripts/run_train.py'],
    capture_output=True, text=True
)

print('STDOUT:')
print(result.stdout)
print()
print('STDERR (log principal):')
print(result.stderr)


## 5. Cargar history.json y plotear curvas

In [ ]:
import json
import matplotlib.pyplot as plt

history_path = f'{PROJECT_ROOT}/experiments/history.json'
with open(history_path) as f:
    data = json.load(f)

history = data['history']
epochs = [h['epoch'] for h in history]

# Extraer series
train_loss = [h['train_loss'] for h in history]
val_loss = [h['val_loss'] for h in history]
train_auc = [h['train_metrics']['auc'] for h in history]
val_auc = [h['val_metrics']['auc'] for h in history]
train_acc = [h['train_metrics']['accuracy'] for h in history]
val_acc = [h['val_metrics']['accuracy'] for h in history]

print(f"Best epoch: {data['best_epoch']} | Best val AUC: {data['best_val_auc']:.4f}")
print(f"Checkpoint: {data['checkpoint_path']}")
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs, train_loss, 'o-', label='train', color='steelblue')
axes[0].plot(epochs, val_loss, 's-', label='val', color='coral')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, train_auc, 'o-', label='train', color='steelblue')
axes[1].plot(epochs, val_auc, 's-', label='val', color='coral')
axes[1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='random')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
axes[1].set_title('AUC')
axes[1].set_ylim(0, 1.05)
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, train_acc, 'o-', label='train', color='steelblue')
axes[2].plot(epochs, val_acc, 's-', label='val', color='coral')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')
axes[2].set_title('Accuracy')
axes[2].set_ylim(0, 1.05)
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f"Baseline training - best val AUC={data['best_val_auc']:.4f} @ epoch {data['best_epoch']}")
plt.tight_layout()
plt.show()


## 6. Tabla detallada de metricas por epoch

In [ ]:
import pandas as pd

rows = []
for h in history:
    rows.append({
        'epoch': h['epoch'],
        'train_loss': h['train_loss'],
        'train_auc': h['train_metrics']['auc'],
        'val_loss': h['val_loss'],
        'val_auc': h['val_metrics']['auc'],
        'val_acc': h['val_metrics']['accuracy'],
        'val_sens': h['val_metrics']['sensitivity'],
        'val_spec': h['val_metrics']['specificity'],
    })

df_hist = pd.DataFrame(rows).round(4)
df_hist


## 7. Siguiente paso

Si val AUC supera 0.70 despues de entrenar, **el baseline es funcional**.
Nos queda cerrar el Sprint 1 con:

1. `src/evaluation/evaluate.py` + `scripts/run_eval.py` - evaluacion final sobre test.
2. `src/explainability/gradcam.py` - Grad-CAM sobre el mejor modelo.

Con eso tendras un entregable completo:
- Checkpoint reproducible con seed=42
- Metricas sobre test set no visto
- Mapas de calor Grad-CAM sobre ejemplos

Vuelve al chat cuando termines esta corrida y pegame el plot o la tabla
de la celda 6.
